In [1]:
import pandas as pd
import numpy as np
import requests
import os
import json

## Flightaware AeroAPI

In [2]:
with open('../apiKeys/apiKeys.json') as f:
    API_KEYS = json.load(f)
    f.close()

In [3]:
AEROAPI_BASE_URL = "https://aeroapi.flightaware.com/aeroapi"

def get_operator_flights(operator_id: str) -> dict:
    url = f"{AEROAPI_BASE_URL}/operators/{operator_id}/flights"
    headers = {"x-apikey": API_KEYS['FA_API_KEY']}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

# Example usage:
data = pd.DataFrame()
for airline in ["UAL", "AAL", "DL", "WN", "AS", "B6"]:
    flights = get_operator_flights(airline)
    flights_df = pd.DataFrame(flights["scheduled"])
    flights_df.to_csv(f"data/{airline}_flights.csv")
    data = pd.concat([data, flights_df]).reset_index(drop=True)
data

/var/folders/mh/b5m2p8fd04x2knz5_hwd1rqw0000gn/T/ipykernel_46568/1058542627.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data, flights_df]).reset_index(drop=True)


,ident,ident_icao,ident_iata,actual_runway_off,actual_runway_on,fa_flight_id,operator,operator_icao,operator_iata,flight_number,...,route,baggage_claim,seats_cabin_business,seats_cabin_coach,seats_cabin_first,gate_origin,gate_destination,terminal_origin,terminal_destination,type
0,UAL664,UAL664,UA664,None,None,UAL664-1781568196-fa-1031p,UAL,UAL,UA,664,...,FARMM289037 FYTTE FYTTE7,None,NaN,NaN,20.0,D56,B9,3,1,Airline
1,UAL1382,UAL1382,UA1382,None,None,UAL1382-1781481797-fa-1015p,UAL,UAL,UA,1382,...,BGOOD5 VIH PWE OVR HANKU FOD MYRRS FYTTE7,None,NaN,NaN,16.0,A16,E14,1,2,Airline
2,UAL704,UAL704,UA704,None,None,UAL704-1781720420-fa-1856p,UAL,UAL,UA,704,...,None,None,NaN,NaN,12.0,None,B19,None,None,Airline
3,UAL2764,UAL2764,UA2764,None,None,UAL2764-1781479456-fa-678p,UAL,UAL,UA,2764,...,KORRN BRWRY LAWGR4,None,NaN,NaN,12.0,23,B19,None,None,Airline
4,UAL1242,UAL1242,UA1242,None,None,UAL1242-1781501365-airline-939p,UAL,UAL,UA,1242,...,NABDA MAM CRP HTOWN3,None,NaN,NaN,NaN,1,E24,None,E,Airline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,JBU1133,JBU1133,B61133,None,None,JBU1133-1781501425-airline-784p,JBU,JBU,B6,1133,...,DDANY3 JIPOD VENDS Y185 RENAH Y355 FIPEK Y355 ...,None,NaN,NaN,NaN,234,A6,C,A,Airline
86,JBU398,JBU398,B6398,None,None,JBU398-1781499571-airline-82p,JBU,JBU,B6,398,...,MONTN2 SEA ZADON ZELAK ZIRAN ZAXBY BLOWS KU21Q...,None,NaN,NaN,NaN,D22,C20,None,None,Airline
87,JBU280,JBU280,B6280,None,None,JBU280-1781506101-airline-1150p,JBU,JBU,B6,280,...,PIBLO UR625 MIROM UR625 ENAMO Y307 JAZZI SAGGY...,None,NaN,NaN,NaN,16,527,None,5,Airline
88,JBU625,JBU625,B6625,None,None,JBU625-1781506413-airline-1524p,JBU,JBU,B6,625,...,WAVEY EMJAY Q167 ZJAAY KALDA Q131 WAALT Y289 B...,None,NaN,NaN,NaN,517,A12,5,None,Airline


## Aircraft Data

In [4]:
with open('aircraft_data/UAL_aircraft_data.json') as f:
    raw = json.load(f)

def _flatten_ual_data(data: dict) -> pd.DataFrame:
    rows = []

    def _process(aircraft_name: str, d: dict):
        for key, val in d.items():
            if not isinstance(val, dict):
                continue
            if any(isinstance(v, dict) for v in val.values()):
                # malformed nesting: key is actually another aircraft variant
                _process(key, val)
            else:
                rows.append({"aircraft": aircraft_name, "cabin_class": key, **val})

    for aircraft, cabins in data.items():
        _process(aircraft, cabins)

    return pd.DataFrame(rows)

ual_aircraft_df = _flatten_ual_data(raw)
ual_aircraft_df

,aircraft,cabin_class,Number of seats,Seat numbers,Exit rows/doors,Seat configuration,Standard seat pitch,Standard seat recline,Limited/Zero seat recline,Seat width,Movable aisle armrests,Fixed bassinet,Entertainment,Wi-Fi,Power outlets,USB ports,Fixed bassinets,Wireless charging,Bulk head seats,Over wing rows
0,B777-200,United First®,28,1A-4L,Front of cabin,2–4–2,"6'4"" (193 cm) sleeping space",180°,N/A,"19"" (48.3 cm)",All rows,No,Seatback entertainment and personal device ent...,True,True,False,NaN,NaN,NaN,NaN
1,B777-200,United Economy Plus®,102,"17A-24L; ABCJKL in rows 25, 26, 39; 40D-G","Rows 17, 39",3-4-3,"35"" (88 cm)","4"" (10 cm)",N/A,"17.1"" (43.4 cm)",All rows,40FG,Personal device entertainment,True,True,True,NaN,NaN,NaN,NaN
2,B777-200,United Economy®,234,"D-G in rows 25, 26; 27A-37L; 40ABCJKL; 41A-53G",Back of cabin,3-4-3,"31"" (78 cm)","3"" (7 cm)",N/A,"17.1"" (43.4 cm)",All rows,No,Personal device entertainment,True,True,False,NaN,NaN,NaN,NaN
3,B777-200ER Version 1,Polaris® Business Class,50,1A-15L,Front of cabin and between rows 8 and 9,1–2–1,"6'6"" (198 cm) sleeping space",180°,N/A,"22"" (55.9 cm)","Rows 2, 4, 6, 8, 10, 12",NaN,Seatback entertainment and personal device ent...,Yes,Yes,Yes,"9A, 9L",NaN,NaN,NaN
4,B777-200ER Version 1,Premium Plus®,24,20A–22L,No,2–4–2,"38"" (96.5 cm)","6"" (15 cm)",N/A,"18.5"" (47 cm)",All aisle seats,NaN,Seatback entertainment and personal device ent...,Yes,Yes,Yes,20DEFG,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,Embraer E175 Version 1,United Economy Plus®,32,7A-16D,No,2-2,"34"" (86 cm)","3"" (7 cm)",N/A,"18.2"" (46 cm)",Rows 8-16,NaN,Personal device entertainment,True,True,False,No,NaN,NaN,09-12
101,Embraer E175 Version 1,United Economy®,26,17A-23B,Back of cabin,2-2,"31"" (78 cm)","2"" (5 cm)",N/A,"18.2"" (46 cm)",All rows,NaN,Personal device entertainment,True,True,False,No,NaN,NaN,No
102,Embraer E175 Version 2,United First®,12,1A-4D,Front of cabin,1-2,"37"" (93 cm)","5"" (12 cm)",N/A,"20"" (50 cm)",Rows 2-4,NaN,Personal device entertainment,True,True,False,No,NaN,1,No
103,Embraer E175 Version 2,United Economy Plus®,16,7A-10D,No,2-2,"34"" (86 cm)","3"" (7 cm)",N/A,"18"" (45 cm)",Rows 8-10,NaN,Personal device entertainment,True,False,False,No,NaN,No,No


In [5]:
data[data['operator_icao'] == 'UAL']

,ident,ident_icao,ident_iata,actual_runway_off,actual_runway_on,fa_flight_id,operator,operator_icao,operator_iata,flight_number,...,route,baggage_claim,seats_cabin_business,seats_cabin_coach,seats_cabin_first,gate_origin,gate_destination,terminal_origin,terminal_destination,type
0,UAL664,UAL664,UA664,None,None,UAL664-1781568196-fa-1031p,UAL,UAL,UA,664,...,FARMM289037 FYTTE FYTTE7,None,NaN,NaN,20.0,D56,B9,3,1,Airline
1,UAL1382,UAL1382,UA1382,None,None,UAL1382-1781481797-fa-1015p,UAL,UAL,UA,1382,...,BGOOD5 VIH PWE OVR HANKU FOD MYRRS FYTTE7,None,NaN,NaN,16.0,A16,E14,1,2,Airline
2,UAL704,UAL704,UA704,None,None,UAL704-1781720420-fa-1856p,UAL,UAL,UA,704,...,None,None,NaN,NaN,12.0,None,B19,None,None,Airline
3,UAL2764,UAL2764,UA2764,None,None,UAL2764-1781479456-fa-678p,UAL,UAL,UA,2764,...,KORRN BRWRY LAWGR4,None,NaN,NaN,12.0,23,B19,None,None,Airline
4,UAL1242,UAL1242,UA1242,None,None,UAL1242-1781501365-airline-939p,UAL,UAL,UA,1242,...,NABDA MAM CRP HTOWN3,None,NaN,NaN,NaN,1,E24,None,E,Airline
5,UAL39,UAL39,UA39,None,None,UAL39-1781466023-fa-2362p,UAL,UAL,UA,39,...,SUMMR2 MCKEY LIBBO BRINY ALCOA 4000N/13000W 44...,None,21.0,NaN,44.0,76A,None,7,3,Airline
6,UAL2458,UAL2458,UA2458,None,None,UAL2458-1781481798-fa-547p,UAL,UAL,UA,2458,...,STYCK8 DOLEY IRW PWE OVR HANKU FOD MYRRS FYTTE7,None,NaN,NaN,20.0,E21,C17,E,1,Airline
7,UAL1560,UAL1560,UA1560,None,None,UAL1560-1781506041-airline-308p,UAL,UAL,UA,1560,...,JAZZI GARIC Q129 AARNN QUART PHLBO4,None,NaN,NaN,16.0,2,BHOLD,M,None,Airline
8,UAL1844,UAL1844,UA1844,None,None,UAL1844-1781489221-fa-978p,UAL,UAL,UA,1844,...,PLMMR3 BURGG Q22 UMBRE QUART PHLBO4,None,NaN,NaN,12.0,T20,A16,None,None,Airline
9,UAL2465,UAL2465,UA2465,None,None,UAL2465-1781481943-fa-648p,UAL,UAL,UA,2465,...,RAYNR BRTMN TAAYZ SQUIB VIO SVM DJB SAVVI TIGRR4,None,NaN,NaN,12.0,C31,A11,1,None,Airline


## Flight Capacity (Available Seat Miles)

Join the United seat-map data onto each scheduled flight to compute **Available Seat Miles (ASM)** per cabin — `seats × route_distance`. ASM is the foundational airline capacity metric and the unit we'll later multiply by fares / load factors to model revenue.

The bridge between the two datasets is a mapping from FlightAware `aircraft_type` codes (`B77W`, `B738`, …) to United's marketing seat-map names (`B777-300ER`, `737-800 Version 1`, …).

In [6]:
import ast

# FlightAware `aircraft_type` code -> United seat-map name (the `aircraft` key in ual_aircraft_df).
# Where United operates multiple configurations of a type, we default to "Version 1".
AIRCRAFT_TYPE_MAP = {
    # ── Wide-bodies ────────────────────────────────────────────────────────
    "B77W": "B777-300ER",              # 777-300ER
    "B772": "B777-200",               # 777-200
    "B77L": "B777-200",               # 777-200LR
    "B788": "B787-8",                 # 787-8 Dreamliner
    "B789": "B787-9 Version 1",       # 787-9
    "B78X": "B787-10",                # 787-10
    "B763": "767-300ER Version 1",    # 767-300ER
    "B764": "767-400ER",              # 767-400ER
    "B752": "757-200",                # 757-200
    "B753": "757-300",                # 757-300
    # ── Narrow-bodies ──────────────────────────────────────────────────────
    "B737": "737-700",                # 737-700
    "B738": "737-800 Version 1",      # 737-800
    "B739": "737-900 Version 1",      # 737-900 / 900ER
    "B38M": "737 MAX 8 Version 1",    # 737 MAX 8
    "B39M": "737 MAX 9 Version 1",    # 737 MAX 9
    "A319": "A319",
    "A320": "A320",
    "A21N": "A321neo",                # A321neo / XLR
    # ── Regional jets ──────────────────────────────────────────────────────
    "CRJ2": "CRJ200",
    "CL65": "CRJ550",
    "CRJ7": "CRJ700",
    "E170": "Embraer E170",
    "E75L": "Embraer E175 Version 1",
    "E75S": "Embraer E175 Version 1",
}


def _iata(cell):
    """Pull the IATA code out of FlightAware's stringified origin/destination dict."""
    try:
        return ast.literal_eval(cell).get("code_iata")
    except (ValueError, SyntaxError, AttributeError, TypeError):
        return None


def compute_flight_capacity(flights_df, aircraft_df, type_map=AIRCRAFT_TYPE_MAP):
    """Join seat-map config onto each flight and compute Available Seat Miles (ASM) per cabin.

    Returns:
        capacity_df: one row per (flight, cabin_class) with seats and ASM = seats * route_distance.
        unmapped:    set of aircraft_type codes present in flights but missing from `type_map`.
    """
    seats = (aircraft_df[["aircraft", "cabin_class", "Number of seats"]]
             .rename(columns={"Number of seats": "seats"})
             .copy())
    seats["seats"] = pd.to_numeric(seats["seats"], errors="coerce")

    f = flights_df.copy()
    f["seatmap_name"] = f["aircraft_type"].map(type_map)

    have_seatmap = set(seats["aircraft"].unique())
    unmapped = set(
        f.loc[~f["seatmap_name"].isin(have_seatmap), "aircraft_type"].dropna().unique()
    )

    id_col = "fa_flight_id" if "fa_flight_id" in f.columns else "ident"
    f = f.rename(columns={id_col: "flight_id"})
    f["origin_iata"] = f["origin"].map(_iata)
    f["destination_iata"] = f["destination"].map(_iata)

    merged = f.merge(seats, left_on="seatmap_name", right_on="aircraft", how="inner")
    merged["ASM"] = merged["seats"] * merged["route_distance"]

    cols = ["flight_id", "ident", "aircraft_type", "seatmap_name",
            "origin_iata", "destination_iata", "route_distance",
            "cabin_class", "seats", "ASM"]
    cols = [c for c in cols if c in merged.columns]
    return merged[cols].reset_index(drop=True), unmapped


In [7]:
# Load UAL flights from the saved CSV (so this runs without a live API call) and compute capacity.
ual_flights = pd.read_csv("data/UAL_flights.csv")

capacity_df, unmapped = compute_flight_capacity(ual_flights, ual_aircraft_df)
print(f"{capacity_df['flight_id'].nunique()} flights joined, "
      f"{len(capacity_df)} (flight x cabin) rows")
if unmapped:
    print(f"Unmapped aircraft_type codes (add to AIRCRAFT_TYPE_MAP): {sorted(unmapped)}")
capacity_df

15 flights joined, 46 (flight x cabin) rows


,flight_id,ident,aircraft_type,seatmap_name,origin_iata,destination_iata,route_distance,cabin_class,seats,ASM
0,UAL664-1781568196-fa-1031p,UAL664,B739,737-900 Version 1,LAS,ORD,1569,United First®,20,31380
1,UAL664-1781568196-fa-1031p,UAL664,B739,737-900 Version 1,LAS,ORD,1569,United Economy Plus®,42,65898
2,UAL664-1781568196-fa-1031p,UAL664,B739,737-900 Version 1,LAS,ORD,1569,United Economy®,117,183573
3,UAL1382-1781481797-fa-1015p,UAL1382,B38M,737 MAX 8 Version 1,STL,ORD,258,United First®,16,4128
4,UAL1382-1781481797-fa-1015p,UAL1382,B38M,737 MAX 8 Version 1,STL,ORD,258,United Economy Plus®,54,13932
5,UAL1382-1781481797-fa-1015p,UAL1382,B38M,737 MAX 8 Version 1,STL,ORD,258,United Economy®,96,24768
6,UAL704-1781720420-fa-1856p,UAL704,A320,A320,SBN,ORD,84,United First®,12,1008
7,UAL704-1781720420-fa-1856p,UAL704,A320,A320,SBN,ORD,84,United Economy Plus®,42,3528
8,UAL704-1781720420-fa-1856p,UAL704,A320,A320,SBN,ORD,84,United Economy®,96,8064
9,UAL2764-1781479456-fa-678p,UAL2764,A319,A319,DSM,DEN,588,United First®,12,7056


In [8]:
# Sanity check: our joined seat totals vs the seats_cabin_* columns FlightAware already provides.
# These only overlap where FlightAware reports seats, but it's a useful cross-check on the join.
flight_seats = (capacity_df.groupby("flight_id")["seats"].sum()
                .rename("seats_joined"))

fa_seats = ual_flights.copy()
fa_seats["fa_total"] = fa_seats[["seats_cabin_first", "seats_cabin_business",
                                 "seats_cabin_coach"]].sum(axis=1, min_count=1)
fa_seats = fa_seats.set_index("fa_flight_id")["fa_total"].rename("seats_from_fa")

check = pd.concat([flight_seats, fa_seats], axis=1).dropna(subset=["seats_joined"])
check

,seats_joined,seats_from_fa
UAL1195-1781481880-fa-769p,179,NaN
UAL1242-1781501365-airline-939p,166,NaN
UAL1271-1781501365-airline-977p,166,16.0
UAL1382-1781481797-fa-1015p,166,16.0
UAL1560-1781506041-airline-308p,166,16.0
UAL1831-1781481856-fa-732p,126,12.0
UAL1844-1781489221-fa-978p,126,12.0
UAL2458-1781481798-fa-547p,179,20.0
UAL2465-1781481943-fa-648p,126,12.0
UAL248-1781464006-fa-1205p,200,20.0


In [9]:
# Insight: ASM by aircraft and cabin, plus each aircraft's premium-vs-economy capacity mix.
PREMIUM_CABINS = ["United First®", "United Business®", "Polaris® Business Class",
                  "United Polaris® Business Class", "United Polaris® business class",
                  "Premium Plus®", "United® Premium Plus"]

asm_by_aircraft = (capacity_df
                   .groupby(["seatmap_name", "cabin_class"])["ASM"]
                   .sum()
                   .unstack(fill_value=0))

cab = capacity_df.copy()
cab["segment"] = np.where(cab["cabin_class"].isin(PREMIUM_CABINS), "premium", "economy")
mix = (cab.groupby(["seatmap_name", "segment"])["ASM"].sum()
       .unstack(fill_value=0))
mix["premium_ASM_share"] = (mix.get("premium", 0) /
                            mix.sum(axis=1)).round(3)
mix = mix.sort_values("premium_ASM_share", ascending=False)
mix

segment,economy,premium,premium_ASM_share
seatmap_name,,,
B787-10,1386946,356330,0.204
737 MAX 9 Version 1,209085,26300,0.112
737-900 Version 1,324519,40820,0.112
A321neo,124200,13800,0.100
737 MAX 8 Version 1,652950,69648,0.096
737-800 Version 1,166500,17760,0.096
737-700,22800,2400,0.095
A319,287850,30300,0.095
A320,11592,1008,0.080
